In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('IMDB Dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'IMDB Dataset.csv'

In [ ]:
df

In [ ]:
df['review'].iloc[1]

In [ ]:
import re
df['clean_text'] = df['review'].apply(lambda x:re.sub("<.*?>","",x))

In [ ]:

df['clean_text'].iloc[1]

In [ ]:
df['clean_text'] = df['clean_text'].apply(lambda x:re.sub(r'[^\w\s]', "",x))

In [ ]:
df['clean_text'].iloc[1]

In [ ]:
df['clean_text'] = df['clean_text'].str.lower()

In [ ]:
df['clean_text'].iloc[1]

In [ ]:
# !pip install nltk

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
df['tokenize_text'] = df['clean_text'].apply(lambda x:word_tokenize(x))

In [ ]:
df['tokenize_text'].iloc[1]

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
from nltk.corpus import stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
stop_words

In [ ]:
df['filtered_text'] = df['tokenize_text'].apply(lambda x:[word for word in x if word not in stop_words])

In [ ]:
len(df['filtered_text'].iloc[1])

In [ ]:
len(df['tokenize_text'].iloc[1])

In [ ]:
from nltk.stem import PorterStemmer

In [ ]:
stem = PorterStemmer()

In [ ]:
df['stem_text'] = df['filtered_text'].apply(lambda x: [stem.stem(word)for word in x])

In [ ]:
df['stem_text'].iloc[1]

In [ ]:
df['filtered_text'].iloc[1]

In [ ]:
from nltk.stem import WordNetLemmatizer

In [ ]:
lemma = WordNetLemmatizer()

In [ ]:
df['lemma_text'] = df['filtered_text'].apply(lambda x: [lemma.lemmatize(word)for word in x])

In [ ]:
df['lemma_text'].iloc[1]

In [ ]:
X = df['stem_text']
y = df['sentiment']

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_tyrain, y_test = train_test_split(X,y, random_state=42, test_size=0.2)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
y_tyrain

In [ ]:
y_test

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer()

In [ ]:
X_train

In [ ]:
X_train = tfidf.fit_transform(X_train.apply(lambda x:''.join(x)))

In [ ]:
X_test = tfidf.transform(X_test.apply(lambda x: "".join(x)))

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

y_train = le.fit_transform(y_tyrain)
y_test = le.transform(y_test)

In [ ]:
from keras.utils import to_categorical

In [ ]:
y_train = to_categorical(y_train, num_classes=2)


In [ ]:
y_train

In [ ]:
X_test.shape

In [ ]:
type(X_train)

In [ ]:
from keras import Sequential

In [ ]:
from keras.layers import Dense

In [ ]:
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(2, activation="sigmoid")
])
    

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs=10)

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import speech_recognition as sr
import pyttsx3

# Load the model and TF-IDF vectorizer
model = joblib.load('model.pkl')
tf_idf_vector = joblib.load('tfidf.pkl')

# Initialize NLTK tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

# Initialize speech recognition and text-to-speech engines
recognizer = sr.Recognizer()
tts_engine = pyttsx3.init()

# Function to convert text to speech
def speak(text):
    tts_engine.say(text)
    tts_engine.runAndWait()

# Function to capture voice input
def get_voice_input():
    with sr.Microphone() as source:
        st.write("Listening for your review...")
        audio = recognizer.listen(source)
        
        try:
            review_text = recognizer.recognize_google(audio)
            st.write(f"Voice Input: {review_text}")
            return review_text
        except sr.UnknownValueError:
            st.write("Sorry, I did not understand that.")
            return None
        except sr.RequestError:
            st.write("Could not request results from Google Speech Recognition.")
            return None

# Function to predict sentiment
def predict_sentiment(review):
    cleaned_review = re.sub('<.*?>', '', review)
    cleaned_review = re.sub(r'[^\w\s]', '', cleaned_review)
    cleaned_review = cleaned_review.lower()
    tokenized_review = word_tokenize(cleaned_review)
    filtered_review = [word for word in tokenized_review if word not in stop_words]
    stemmed_review = [stemmer.stem(word) for word in filtered_review]
    tfidf_review = tf_idf_vector.transform([' '.join(stemmed_review)])
    sentiment_prediction = model.predict(tfidf_review)
    
    if sentiment_prediction > 0.6:  # Adjust threshold as needed
        return "Positive"
    else:
        return "Negative"

# Streamlit UI
st.title('Sentiment Analysis')

# Choice for user input method: Text or Voice
input_method = st.radio("Choose your input method:", ('Type', 'Speak'))

if input_method == 'Type':
    # Text input by user
    review_to_predict = st.text_area('Enter your review here:')
    if st.button('Predict Sentiment'):
        predicted_sentiment = predict_sentiment(review_to_predict)
        st.write("Predicted Sentiment:", predicted_sentiment)
        speak(f"The predicted sentiment is {predicted_sentiment}")

elif input_method == 'Speak':
    # Voice input by user
    if st.button('Speak your review'):
        voice_review = get_voice_input()
        
        if voice_review:
            predicted_sentiment = predict_sentiment(voice_review)
            st.write("Predicted Sentiment:", predicted_sentiment)
            speak(f"The predicted sentiment is {predicted_sentiment}")


In [ ]:
!streamlit run NLP.ipynb

In [ ]:
!ipynb-py-convert NLP.ipynb NLP.py

In [ ]:
!pip install ipynb-py-convert

In [ ]:
pip install streamlit speechrecognition pyttsx3 joblib
